In [1]:
pip install pyspark==3.4.1 findspark

In [2]:
# This cell is now redundant as JAVA_HOME is set in the Spark initialization cell.
# Keeping it empty or deleting it to avoid confusion.


In [3]:
# This cell is now redundant as imports are handled in the Spark initialization cell.
# Keeping it empty or deleting it to avoid confusion.


In [5]:
# Install Java (if not already present or for specific version)
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
# Removed explicit SPARK_HOME setting to let findspark.init() auto-discover

import findspark
findspark.init()

from pyspark import SparkConf, SparkContext
from pyspark.sql import SQLContext

conf = SparkConf().set('spark.ui.port', '4050').setAppName("films").setMaster("local[2]")
sc = SparkContext.getOrCreate(conf=conf)
sqlContext = SQLContext(sc)
#sc.stop()

In [7]:
house_df = sqlContext.read.format('com.databricks.spark.csv').options(header='true', inferschema='true').load('/content/Boston.csv')
house_df.show()

+---+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+
|_c0|   crim|  zn|indus|chas|  nox|   rm|  age|   dis|rad|tax|ptratio| black|lstat|medv|
+---+-------+----+-----+----+-----+-----+-----+------+---+---+-------+------+-----+----+
|  1|0.00632|18.0| 2.31|   0|0.538|6.575| 65.2|  4.09|  1|296|   15.3| 396.9| 4.98|24.0|
|  2|0.02731| 0.0| 7.07|   0|0.469|6.421| 78.9|4.9671|  2|242|   17.8| 396.9| 9.14|21.6|
|  3|0.02729| 0.0| 7.07|   0|0.469|7.185| 61.1|4.9671|  2|242|   17.8|392.83| 4.03|34.7|
|  4|0.03237| 0.0| 2.18|   0|0.458|6.998| 45.8|6.0622|  3|222|   18.7|394.63| 2.94|33.4|
|  5|0.06905| 0.0| 2.18|   0|0.458|7.147| 54.2|6.0622|  3|222|   18.7| 396.9| 5.33|36.2|
|  6|0.02985| 0.0| 2.18|   0|0.458| 6.43| 58.7|6.0622|  3|222|   18.7|394.12| 5.21|28.7|
|  7|0.08829|12.5| 7.87|   0|0.524|6.012| 66.6|5.5605|  5|311|   15.2| 395.6|12.43|22.9|
|  8|0.14455|12.5| 7.87|   0|0.524|6.172| 96.1|5.9505|  5|311|   15.2| 396.9|19.15|27.1|
|  9|0.21124|12.5| 7.

In [8]:
## Printing schema
house_df.printSchema()


root
 |-- _c0: integer (nullable = true)
 |-- crim: double (nullable = true)
 |-- zn: double (nullable = true)
 |-- indus: double (nullable = true)
 |-- chas: integer (nullable = true)
 |-- nox: double (nullable = true)
 |-- rm: double (nullable = true)
 |-- age: double (nullable = true)
 |-- dis: double (nullable = true)
 |-- rad: integer (nullable = true)
 |-- tax: integer (nullable = true)
 |-- ptratio: double (nullable = true)
 |-- black: double (nullable = true)
 |-- lstat: double (nullable = true)
 |-- medv: double (nullable = true)



In [9]:
## Descriptive analysis
house_df.toPandas()

,_c0,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,black,lstat,medv
0,1,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,2,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,3,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,4,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,5,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,502,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273,21.0,391.99,9.67,22.4
502,503,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273,21.0,396.90,9.08,20.6
503,504,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273,21.0,396.90,5.64,23.9
504,505,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273,21.0,393.45,6.48,22.0


In [10]:
from pyspark.ml.feature import VectorAssembler
vectorAssembler = VectorAssembler(inputCols = ['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'black', 'lstat'], outputCol = 'features')
vhouse_df = vectorAssembler.transform(house_df)
vhouse_df = vhouse_df.select(['features', 'medv'])
vhouse_df.show(3)

+--------------------+----+
|            features|medv|
+--------------------+----+
|[0.00632,18.0,2.3...|24.0|
|[0.02731,0.0,7.07...|21.6|
|[0.02729,0.0,7.07...|34.7|
+--------------------+----+
only showing top 3 rows



In [11]:
splits = vhouse_df.randomSplit([0.7, 0.3])
train_df = splits[0]
test_df = splits[1]
#train_df,test_df=vhouse_df.randomSplit([0.7,0.3])

In [12]:
from pyspark.ml.regression import LinearRegression
lr = LinearRegression(featuresCol = 'features', labelCol='medv', maxIter=10)
lr_model = lr.fit(train_df)
print("Coefficients: " + str(lr_model.coefficients))
print("Intercept: " + str(lr_model.intercept))

Coefficients: [-0.1301189782735196,0.03288235674710829,-0.03160931970850113,1.5747126940117278,-21.23655619337029,3.538486015338964,-0.0038648580288316676,-1.6788364104137912,0.29944925411524725,-0.009009841770674421,-1.143354524131212,0.006105760372306774,-0.5186625801819014]
Intercept: 45.508787071702116


In [13]:
trainingSummary = lr_model.summary
print("RMSE: %f" % trainingSummary.rootMeanSquaredError)
print("r2: %f" % trainingSummary.r2)

RMSE: 4.885045
r2: 0.731309


In [14]:
lr_predictions = lr_model.transform(test_df)
lr_predictions.select("prediction","medv","features").show(5)
from pyspark.ml.evaluation import RegressionEvaluator
lr_evaluator = RegressionEvaluator(predictionCol="prediction", \
                 labelCol="medv",metricName="r2")
print("R Squared (R2) on test data = %g" % lr_evaluator.evaluate(lr_predictions))

+------------------+----+--------------------+
|        prediction|medv|            features|
+------------------+----+--------------------+
| 30.72832678267975|24.0|[0.00632,18.0,2.3...|
|27.671194447237195|22.0|[0.01096,55.0,2.2...|
|30.420919021289457|32.7|[0.01301,35.0,1.5...|
|29.752936552707858|35.4|[0.01311,90.0,1.2...|
| 23.39295735713509|30.1|[0.01709,90.0,2.0...|
+------------------+----+--------------------+
only showing top 5 rows

R Squared (R2) on test data = 0.739378


In [15]:
print("numIterations: %d" % trainingSummary.totalIterations)
print("objectiveHistory: %s" % str(trainingSummary.objectiveHistory))
trainingSummary.residuals.show()


numIterations: 0
objectiveHistory: [0.0]
+-------------------+
|          residuals|
+-------------------+
| 1.0119587179829779|
|  4.103049947840219|
|   9.27699773311307|
|-0.6873366475867329|
| -2.729001539781585|
|-2.2090547838033494|
|   6.71166793375145|
|  7.146650631824791|
|   2.65217109970213|
| -1.246724887264719|
|  10.46313369972042|
| 0.8326971878436318|
| 0.6760841150433166|
|   5.16047593474908|
|-1.2206210935752537|
| -6.547655682199025|
|  4.154133840650324|
|-0.0528526005535106|
|-3.9630705297748428|
|  3.420588685012646|
+-------------------+
only showing top 20 rows



In [16]:
predictions = lr_model.transform(test_df)
predictions.select("prediction","medv","features").show()

+------------------+----+--------------------+
|        prediction|medv|            features|
+------------------+----+--------------------+
| 30.72832678267975|24.0|[0.00632,18.0,2.3...|
|27.671194447237195|22.0|[0.01096,55.0,2.2...|
|30.420919021289457|32.7|[0.01301,35.0,1.5...|
|29.752936552707858|35.4|[0.01311,90.0,1.2...|
| 23.39295735713509|30.1|[0.01709,90.0,2.0...|
| 42.68451322872109|50.0|[0.02009,95.0,2.6...|
|  27.5084416756168|23.9|[0.02543,55.0,3.7...|
|25.148150428323714|21.6|[0.02731,0.0,7.07...|
| 21.41251040968494|26.6|[0.02899,40.0,1.2...|
|22.008435200710004|20.6|[0.03306,0.0,5.19...|
|33.606886120218334|34.9|[0.03359,75.0,2.9...|
|32.563957686980665|28.5|[0.03502,80.0,4.9...|
|41.568329504704444|48.5|[0.0351,95.0,2.68...|
|28.162886427629832|22.0|[0.03932,0.0,3.41...|
|20.235086506463066|21.1|[0.03961,0.0,5.19...|
| 26.83210530219471|24.8|[0.04297,52.5,5.3...|
|12.681616644249736|18.2|[0.04301,80.0,1.9...|
| 23.96068575673778|20.5|[0.04337,21.0,5.6...|
|25.698143459

## Decision tree regression

In [17]:
from pyspark.ml.regression import DecisionTreeRegressor
dt = DecisionTreeRegressor(featuresCol ='features', labelCol = 'medv')
dt_model = dt.fit(train_df)
dt_predictions = dt_model.transform(test_df)
dt_evaluator = RegressionEvaluator(
    labelCol="medv", predictionCol="prediction", metricName="rmse")
rmse = dt_evaluator.evaluate(dt_predictions)
print("Root Mean Squared Error (RMSE) on test data = %g" % rmse)

Root Mean Squared Error (RMSE) on test data = 4.27765


In [18]:
from pyspark.ml.regression import DecisionTreeRegressor
dt = DecisionTreeRegressor(featuresCol ='features', labelCol = 'medv')
dt_model = dt.fit(train_df)
dt_predictions = dt_model.transform(test_df)
dt_evaluator = RegressionEvaluator(
    labelCol="medv", predictionCol="prediction", metricName="r2")
r2 = dt_evaluator.evaluate(dt_predictions)
print("R2 on test data = %g" % r2)

R2 on test data = 0.755643


In [19]:
lr_evaluator.evaluate(dt_predictions)

0.755642907522552

In [20]:
 dt_model.featureImportances

SparseVector(13, {0: 0.1068, 4: 0.0347, 5: 0.6156, 6: 0.0094, 7: 0.0688, 9: 0.0058, 10: 0.0052, 11: 0.0062, 12: 0.1475})

In [21]:
house_df.take(1)

[Row(_c0=1, crim=0.00632, zn=18.0, indus=2.31, chas=0, nox=0.538, rm=6.575, age=65.2, dis=4.09, rad=1, tax=296, ptratio=15.3, black=396.9, lstat=4.98, medv=24.0)]

## Gradient-boosted tree regression

In [22]:
from pyspark.ml.regression import GBTRegressor
gbt = GBTRegressor(featuresCol = 'features', labelCol = 'medv', maxIter=10)
gbt_model = gbt.fit(train_df)
gbt_predictions = gbt_model.transform(test_df)
gbt_predictions.select('prediction', 'medv', 'features').show(5)

+------------------+----+--------------------+
|        prediction|medv|            features|
+------------------+----+--------------------+
| 23.56412844532784|24.0|[0.00632,18.0,2.3...|
|20.537868219908823|22.0|[0.01096,55.0,2.2...|
| 35.67816291668635|32.7|[0.01301,35.0,1.5...|
| 37.43097911942517|35.4|[0.01311,90.0,1.2...|
| 33.68861286221358|30.1|[0.01709,90.0,2.0...|
+------------------+----+--------------------+
only showing top 5 rows



In [23]:
gbt_evaluator = RegressionEvaluator(
    labelCol="medv", predictionCol="prediction", metricName="rmse")
rmse = gbt_evaluator.evaluate(gbt_predictions)
print("Root Mean Squared Error (RMSE) on test data = %g" % rmse)

Root Mean Squared Error (RMSE) on test data = 3.97134


In [24]:
gbt_evaluator = RegressionEvaluator(
    labelCol="medv", predictionCol="prediction", metricName="r2")
r2 = gbt_evaluator.evaluate(gbt_predictions)
print("R2 Score on test data = %g" % r2)

R2 Score on test data = 0.789385
